# OOP Week 2 -- Composition: DataSource & Dataset

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Week 1 (Classes & Objects)
**Focus:** composition, has-a relationships, wrapping data in objects

---

## Learning Objectives

1. Explain composition ('has-a') relationships
2. Build a `Dataset` class that wraps raw data with metadata
3. Build a `DataSource` that creates `Dataset` objects
4. Understand why we wrap data in objects instead of using raw dicts/lists
5. Draw composition diagrams

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Section 1: What is Composition?

**Composition** means one object **contains** another object. We say object A 'has-a' object B.

### Real-World Examples

- A **Car** has-a Engine, has-a Transmission, has-a Battery
- A **Kitchen** has-a Oven, has-a Refrigerator, has-a Sink
- A **Robot** has-a MotorController, has-a SensorArray, has-a Battery
- A **Pipeline** has-a DataSource, has-a Cleaner, has-a Analyzer

Composition is the **most common** relationship in OOP. It is how you build complex systems from simple parts.

### Composition vs Inheritance

| Composition (has-a) | Inheritance (is-a) |
|--------------------|-----------------|
| Car HAS-A Engine | ElectricCar IS-A Car |
| Pipeline HAS-A Cleaner | RangeCleaner IS-A Cleaner |
| Flexible, easy to swap parts | Rigid hierarchy |
| **Preferred in modern Python** | Use sparingly |

We will focus on composition for the next several weeks.

---
### Composition: Pipeline has-a DataSource, Cleaner, Analyzer

```
+------------------+       +------------------+
|    Pipeline      |       |    DataSource     |
+------------------+  has  +------------------+
| - source --------+------>| - path: str      |
| - cleaner -------+--+    | + load() -> Dataset
| - analyzer ------+-+|    +------------------+
+------------------+ ||    
| + run()          | ||    +------------------+
+------------------+ |+--->|    Cleaner        |
                     |     +------------------+
                     |     | - min_val: float |
                     |     | + clean(data)    |
                     |     +------------------+
                     |     
                     +---->+------------------+
                           |    Analyzer       |
                           +------------------+
                           | - column: str    |
                           | + analyze(data)  |
                           +------------------+
```

---
## Section 2: The Dataset Class

Right now your data is a plain list of dictionaries. That works, but it has problems:

- No metadata (where did it come from? how many rows?)
- No helper methods (want column names? write code every time)
- No validation (is the data even valid?)

A `Dataset` class wraps the raw data and adds all of this.

### Example 1: Basic Dataset

In [ ]:
class Dataset:
    """A structured container for tabular data."""

    def __init__(self, rows, source_path="unknown"):
        self.rows = rows
        self.source_path = source_path
        self.n_rows = len(rows)
        self.columns = list(rows[0].keys()) if rows else []

    def get_column(self, name):
        """Extract all values for a given column."""
        return [row.get(name) for row in self.rows]

    def head(self, n=5):
        """Return first n rows."""
        return self.rows[:n]

    def describe(self):
        """Print a summary of the dataset."""
        print("Dataset Summary")
        print("  Source:  " + self.source_path)
        print("  Rows:   " + str(self.n_rows))
        print("  Columns: " + str(self.columns))

    def __len__(self):
        return self.n_rows

    def __str__(self):
        return "Dataset(" + str(self.n_rows) + " rows, columns=" + str(self.columns) + ")"


# Create a Dataset
raw = [
    {"id": 1, "value": 25.0, "status": "ok"},
    {"id": 2, "value": 30.0, "status": "ok"},
    {"id": 3, "value": 88.0, "status": "warning"},
    {"id": 4, "value": -5.0, "status": "error"},
]
ds = Dataset(raw, "data/raw/sensors.csv")

print(ds)
ds.describe()
print("Values:", ds.get_column("value"))
print("First 2:", ds.head(2))

**Expected Output:**
```
Dataset(4 rows, columns=['id', 'value', 'status'])
Dataset Summary
  Source:  data/raw/sensors.csv
  Rows:   4
  Columns: ['id', 'value', 'status']
Values: [25.0, 30.0, 88.0, -5.0]
First 2: [{'id': 1, 'value': 25.0, 'status': 'ok'}, {'id': 2, 'value': 30.0, 'status': 'ok'}]
```

### Example 2: Adding Filtering to Dataset

In [ ]:
class Dataset:
    """Enhanced Dataset with filtering."""

    def __init__(self, rows, source_path="unknown"):
        self.rows = rows
        self.source_path = source_path
        self.n_rows = len(rows)
        self.columns = list(rows[0].keys()) if rows else []

    def get_column(self, name):
        return [row.get(name) for row in self.rows]

    def filter_rows(self, column, condition):
        """Return a NEW Dataset with only rows meeting condition.

        Does NOT modify the original dataset.
        """
        kept = [row for row in self.rows if condition(row.get(column))]
        return Dataset(kept, self.source_path)

    def __len__(self):
        return self.n_rows

    def __str__(self):
        return "Dataset(" + str(self.n_rows) + " rows)"


ds = Dataset(raw, "sensors.csv")
print("Original:", ds)

# Filter: keep only positive values
positive = ds.filter_rows("value", lambda v: isinstance(v, (int, float)) and v > 0)
print("Positive:", positive)
print("Positive values:", positive.get_column("value"))

# Original is unchanged!
print("Original still has:", len(ds), "rows")

**Expected Output:**
```
Original: Dataset(4 rows)
Positive: Dataset(3 rows)
Positive values: [25.0, 30.0, 88.0]
Original still has: 4 rows
```

---
### Design Decision: Why filter_rows returns a NEW Dataset

Notice that `filter_rows` does not modify `self.rows`. It creates and returns a **new** Dataset. This is called being **immutable** (or at least non-destructive).

Why? Because you might want to:
- Apply different filters to the same original data
- Compare before and after
- Debug by looking at the original

**Rule of thumb:** Methods that transform data should return new objects, not modify the original. Methods that update state (like `add_reading`) are the exception.

---
## Section 3: The DataSource Class

The DataSource is responsible for loading data from an external source and returning a Dataset. This separates the 'where data comes from' logic from the 'what data looks like' logic.

In [ ]:
class DataSource:
    """Loads raw data and returns a Dataset."""

    def __init__(self, path, required_columns=None):
        self.path = path
        self.required_columns = required_columns or []

    def load(self):
        """Load data and return as a Dataset."""
        # Simulate loading from file
        raw = [
            {"id": 1, "value": 25.0, "status": "ok"},
            {"id": 2, "value": 30.0, "status": "ok"},
            {"id": 3, "value": 88.0, "status": "warning"},
            {"id": 4, "value": -5.0, "status": "error"},
        ]

        # Validate required columns
        if raw and self.required_columns:
            actual = set(raw[0].keys())
            missing = set(self.required_columns) - actual
            if missing:
                raise ValueError("Missing columns: " + str(missing))

        dataset = Dataset(raw, self.path)
        print("Loaded: " + str(dataset))
        return dataset


# DataSource CREATES a Dataset (composition in action)
source = DataSource("data/raw/sensors.csv", required_columns=["id", "value"])
dataset = source.load()

print("Rows:", len(dataset))
print("Columns:", dataset.columns)
print("Values:", dataset.get_column("value"))

**Expected Output:**
```
Loaded: Dataset(4 rows)
Rows: 4
Columns: ['id', 'value', 'status']
Values: [25.0, 30.0, 88.0, -5.0]
```

---
## Section 4: Composing Components Together

Now let us compose DataSource, Dataset, and a simple Cleaner into a mini-pipeline. Each component does one job and passes its output to the next.

In [ ]:
class SimpleCleaner:
    """Cleans a Dataset by filtering rows."""

    def __init__(self, value_column, min_val, max_val):
        self.value_column = value_column
        self.min_val = min_val
        self.max_val = max_val
        self.drop_count = 0

    def clean(self, dataset):
        """Return a new clean Dataset."""
        clean_rows = []
        self.drop_count = 0
        for row in dataset.rows:
            val = row.get(self.value_column)
            if isinstance(val, (int, float)) and self.min_val <= val <= self.max_val:
                clean_rows.append(row)
            else:
                self.drop_count += 1
        result = Dataset(clean_rows, dataset.source_path)
        print("Cleaned: " + str(len(dataset)) + " -> " + str(len(result))
              + " (dropped " + str(self.drop_count) + ")")
        return result


# Compose the pipeline
source = DataSource("data/raw/sensors.csv")
cleaner = SimpleCleaner("value", min_val=0, max_val=100)

# Run
dataset = source.load()
clean_dataset = cleaner.clean(dataset)

print()
print("Original values:", dataset.get_column("value"))
print("Clean values:   ", clean_dataset.get_column("value"))

**Expected Output:**
```
Loaded: Dataset(4 rows)
Cleaned: 4 -> 3 (dropped 1)

Original values: [25.0, 30.0, 88.0, -5.0]
Clean values:    [25.0, 30.0, 88.0]
```

---
### Procedural vs OOP: Loading and Cleaning Data

The OOP version is **self-documenting**. The Dataset object knows where it came from, what columns it has, and how many rows. In the procedural version, that metadata is lost -- it is just a plain list.

**Procedural approach (what you did in CP1/CP2):**

In [ ]:
# PROCEDURAL
def load_data(path):
    return [{"id": 1, "value": 25}, {"id": 2, "value": -5}]

def clean_data(data, col, lo, hi):
    return [r for r in data if lo <= r.get(col, 0) <= hi]

raw = load_data("data.csv")
clean = clean_data(raw, "value", 0, 100)
print(len(clean), "rows")
# No metadata! Where did it come from? What columns?

**OOP approach (what we are learning now):**

In [ ]:
# OOP
source = DataSource("data.csv")
dataset = source.load()   # returns a Dataset object
print(dataset)            # knows its own metadata
print(dataset.columns)    # self-documenting

clean = cleaner.clean(dataset)  # returns new Dataset
print(clean)              # metadata preserved

---
### Common Mistake: Modifying the original dataset

The code below has a bug. Can you spot it before reading the fix?

In [ ]:
class BadCleaner:
    def clean(self, dataset):
        # BUG: modifies the original!
        i = 0
        while i < len(dataset.rows):
            if dataset.rows[i].get("value", 0) < 0:
                dataset.rows.pop(i)
            else:
                i += 1
        return dataset

ds = Dataset([{"value": 10}, {"value": -5}, {"value": 20}])
print("Before:", len(ds), "rows")
cleaned = BadCleaner().clean(ds)
print("After: ", len(ds), "rows")  # OOPS: original changed!

**What goes wrong:** The BadCleaner uses `pop()` to remove items from the original list. Since lists are mutable, this modifies the original Dataset. Always create a NEW list/Dataset when cleaning.

**The fix:**

In [ ]:
class GoodCleaner:
    def clean(self, dataset):
        # FIXED: create new list, do not modify original
        kept = [r for r in dataset.rows if r.get("value", 0) >= 0]
        return Dataset(kept, dataset.source_path)

ds = Dataset([{"value": 10}, {"value": -5}, {"value": 20}])
print("Before:", len(ds), "rows")
cleaned = GoodCleaner().clean(ds)
print("After: ", len(ds), "rows")  # Original unchanged!
print("Clean: ", len(cleaned), "rows")

---
### Try It!

Create a `Dataset` that holds student grade data:
```python
grades = [
    {"name": "Alice", "score": 92},
    {"name": "Bob", "score": 67},
    {"name": "Charlie", "score": 45},
    {"name": "Diana", "score": 88},
]
```
1. Create the Dataset
2. Use `get_column` to extract all scores
3. Create a `GradeCleaner` that drops scores below 50
4. Verify the original dataset is unchanged

In [ ]:
# YOUR CODE HERE


---
## Section 5: More Worked Examples

### Example 3: Dataset with Type Coercion

Real data often has string values that need to be converted. Let us add a method that handles this.

In [ ]:
class SmartDataset(Dataset):
    """Dataset that can coerce column types."""

    def get_numeric_column(self, name):
        """Get a column, converting strings to floats where possible."""
        result = []
        for row in self.rows:
            val = row.get(name)
            if isinstance(val, (int, float)):
                result.append(float(val))
            elif isinstance(val, str):
                try:
                    result.append(float(val))
                except ValueError:
                    result.append(None)  # mark as missing
            else:
                result.append(None)
        return result


messy = SmartDataset([
    {"id": 1, "value": "25.5"},
    {"id": 2, "value": 30},
    {"id": 3, "value": "abc"},
    {"id": 4, "value": None},
], "messy.csv")

print("Raw:", messy.get_column("value"))
print("Numeric:", messy.get_numeric_column("value"))

**Expected Output:**
```
Raw: ['25.5', 30, 'abc', None]
Numeric: [25.5, 30.0, None, None]
```

### Example 4: Chaining Operations

Because each method returns a new Dataset, we can chain operations together.

In [ ]:
# Chaining: load -> filter -> filter -> get_column
raw = Dataset([
    {"id": 1, "value": 25.0, "category": "A"},
    {"id": 2, "value": -5.0, "category": "B"},
    {"id": 3, "value": 88.0, "category": "A"},
    {"id": 4, "value": 42.0, "category": "B"},
    {"id": 5, "value": 200.0, "category": "A"},
], "raw.csv")

# Chain: keep positive -> keep < 100 -> get values
result = (raw
    .filter_rows("value", lambda v: isinstance(v, (int, float)) and v > 0)
    .filter_rows("value", lambda v: v < 100))

print("Original:", len(raw), "rows")
print("After chain:", len(result), "rows")
print("Values:", result.get_column("value"))

**Expected Output:**
```
Original: 5 rows
After chain: 3 rows
Values: [25.0, 88.0, 42.0]
```

---
### Debugging Tip: TypeError: 'NoneType' object is not iterable

This happens when a method returns None instead of a Dataset. Check that every method that should return a Dataset actually has a `return` statement. Common mistake:

```python
def filter_rows(self, column, condition):
    kept = [r for r in self.rows if condition(r.get(column))]
    Dataset(kept)  # BUG: missing 'return'!
```

Fix: `return Dataset(kept, self.source_path)`

---
### Try It!

Create a `merge_datasets(ds1, ds2)` function that combines two Datasets into one new Dataset. The datasets must have the same columns. Raise ValueError if they do not.

Test with:
```python
ds1 = Dataset([{"x": 1}, {"x": 2}])
ds2 = Dataset([{"x": 3}, {"x": 4}])
merged = merge_datasets(ds1, ds2)
print(len(merged))  # should be 4
```

In [ ]:
# YOUR CODE HERE


---
### Common Mistake: Shallow copy trap with nested data

The code below has a bug. Can you spot it before reading the fix?

In [ ]:
# BUG: rows share references to the same dicts!
original = Dataset([{"value": [1, 2, 3]}])
copy_rows = original.rows[:]  # shallow copy of list
copy_rows[0]["value"].append(999)  # modifies original too!

print("Original:", original.rows[0]["value"])
print("Oops! 999 leaked into the original.")

**What goes wrong:** When your data contains nested mutable objects (lists, dicts), a shallow copy (slicing, list()) still shares the inner objects. Use `copy.deepcopy()` for truly independent copies.

**The fix:**

In [ ]:
import copy

original = Dataset([{"value": [1, 2, 3]}])
copy_rows = copy.deepcopy(original.rows)  # deep copy!
copy_rows[0]["value"].append(999)

print("Original:", original.rows[0]["value"])  # [1, 2, 3]
print("Copy:", copy_rows[0]["value"])  # [1, 2, 3, 999]
print("Original is safe!")

---
### Design Decision: When to use Dataset vs plain list

**Use Dataset** when:
- You need metadata (source path, column names, row count)
- You want helper methods (get_column, filter, describe)
- Data will pass through multiple components
- You want type safety (only Datasets enter the pipeline)

**Use plain list** when:
- Quick throwaway computation
- Simple scripts with no pipeline
- Performance-critical inner loops

---
### Try It!

Add a `sample(n)` method to Dataset that returns a NEW Dataset with `n` random rows. Use `import random` and `random.sample(self.rows, n)`. Handle the case where `n > len(self.rows)`.

In [ ]:
# YOUR CODE HERE


---
## Build from Scratch Exercise

This exercise tests whether you truly understand this week's concepts. Complete it without looking at the examples above.

In [ ]:
# BUILD FROM SCRATCH:
# Build a `ShoppingCart` (has-a list of `CartItem` objects). CartItem has name, price, quantity. Cart has add_item(), total(), remove_item(). Demonstrate composition.

# YOUR CODE HERE


In [ ]:
# TEST your build-from-scratch code:

# YOUR TESTS HERE


---
## Connect the Dots

How does this week's concept connect to previous weeks?

In [ ]:
# How does composition (has-a) relate to the classes you built in Week 1?

# YOUR ANSWER (as comments or code):


---
## Real-World Spotting

OOP patterns are everywhere in real software. Can you spot them?

In [ ]:
# Think about a smartphone. List 5 'has-a' composition relationships (e.g., Phone has-a Camera, Camera has-a Sensor).

# YOUR ANSWER:


---
## Diagram It

Draw an ASCII class diagram for the main classes from this week. Include:
- Class names
- Key attributes
- Key methods
- Relationships (has-a, is-a)

In [ ]:
# Draw your ASCII diagram here:
# +------------------+
# |   ClassName      |
# +------------------+
# | - attribute      |
# +------------------+
# | + method()       |
# +------------------+

# YOUR DIAGRAM:


---
## Key Vocabulary

| Term | Definition |
|------|------------|
| **Composition** | One object contains another ('has-a') |
| **Has-a** | A relationship where one object owns another |
| **Is-a** | An inheritance relationship (subclass IS-A parent) |
| **Dataset** | An object that wraps raw data with metadata |
| **DataSource** | An object that loads data and creates Datasets |
| **Immutable** | Cannot be changed after creation |
| **Non-destructive** | Returns new objects instead of modifying originals |

---
## Recap Exercise

Without looking at the code above, try to:

In [ ]:
# 1. Write one class from this week FROM MEMORY
#    (it does not need to be perfect)

# YOUR CODE HERE


# 2. Create an instance and call at least one method

# YOUR CODE HERE


# 3. Write one test for your class

# YOUR CODE HERE


---
## What to Review Before Next Week

Before the next session, make sure you can:

1. Explain this week's main concept in your own words
2. Write a simple example from memory
3. Identify this pattern in existing code
4. Explain WHY this pattern is useful (not just HOW)

---
## Mini-Quiz

In [ ]:
# Q1: What is composition?
# Answer: 

# Q2: What is the difference between 'has-a' and 'is-a'?
# Answer: 

# Q3: Why should clean() return a NEW dataset instead of modifying the original?
# Answer: 

# Q4: What does the DataSource create and return?
# Answer: 

# Q5: Draw (in comments) a composition diagram for a school:
#      School has-a ... has-a ... 
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)